# 🏠 Appliances Energy Prediction — Complete Data Mining Pipeline

**Run this notebook top-to-bottom to execute the full pipeline and see all graphs inline.**

| Section | Technique |
|---------|----------|
| 1 | Data Preprocessing & Descriptive Statistics |
| 2 | Regression Analysis (Simple LM · Multiple LM · KNN · Decision Tree) |
| 3 | Classification Analysis (Logistic · KNN · DT · Naive Bayes · SVM · Perceptron · MLP) |
| 4 | Clustering Analysis (K-Means · Hierarchical · GMM / EM) |

> **Dataset**: `energydata_complete.csv` — 10-minute interval recordings of appliance energy use (Wh) along with indoor temperature/humidity sensors, outdoor weather, and time variables.

## ⚙️ 0. Setup — Imports & Display Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid', palette='muted')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

print('✅ All libraries imported successfully.')


---
## 📊 1. Data Preprocessing & Descriptive Statistics

**Steps:** Load CSV → Descriptive stats → IQR outlier detection → Feature engineering (NSM, WeekStatus, Day_of_week) → Drop noise (rv1/rv2) → Standard scaling → One-hot encoding → Binary target creation → Stratified 75/25 split

In [ ]:
CSV_PATH = 'energydata_complete.csv'
assert os.path.exists(CSV_PATH), f"❌ '{CSV_PATH}' not found in the current directory."

df_raw = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head(3)


In [ ]:
# Descriptive Statistics
numerical_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
stats_rows = []
for col in numerical_cols:
    stats_rows.append({
        'Feature': col,
        'Mean':    round(df_raw[col].mean(), 4),
        'Median':  round(df_raw[col].median(), 4),
        'Mode':    round(df_raw[col].mode()[0], 4),
        'Std Dev': round(df_raw[col].std(), 4),
        'Min':     round(df_raw[col].min(), 4),
        'Max':     round(df_raw[col].max(), 4),
    })
stats_df = pd.DataFrame(stats_rows)
print('📋 Descriptive Statistics:')
display(stats_df)


In [ ]:
# IQR Outlier Detection on 'Appliances'
q1  = df_raw['Appliances'].quantile(0.25)
q3  = df_raw['Appliances'].quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df_raw[(df_raw['Appliances'] < lo) | (df_raw['Appliances'] > hi)]
print(f'IQR bounds: [{lo:.2f}, {hi:.2f}]')
print(f'Outliers: {len(outliers):,} rows ({len(outliers)/len(df_raw)*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_raw['Appliances'], bins=80, color='#4e79a7', edgecolor='white', linewidth=0.4)
axes[0].axvline(hi, color='red', linestyle='--', linewidth=1.8, label=f'IQR Upper ({hi:.0f} Wh)')
axes[0].set_xlabel('Appliances Energy (Wh)'); axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Appliances Energy', fontweight='bold')
axes[0].legend()
axes[1].boxplot(df_raw['Appliances'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4e79a7', color='navy'),
                medianprops=dict(color='orange', linewidth=2))
axes[1].set_ylabel('Appliances Energy (Wh)')
axes[1].set_title('Boxplot — Outlier Summary', fontweight='bold')
axes[1].set_xticks([])
plt.tight_layout(); plt.show()


In [ ]:
# Feature Engineering & Preprocessing
df = df_raw.copy()
df['date']        = pd.to_datetime(df['date'])
df['NSM']         = df['date'].dt.hour * 3600 + df['date'].dt.minute * 60 + df['date'].dt.second
df['WeekStatus']  = df['date'].dt.dayofweek.map(lambda x: 'Weekend' if x >= 5 else 'Weekday')
df['Day_of_week'] = df['date'].dt.day_name()
for col in ['rv1', 'rv2']:
    if col in df.columns: df.drop(columns=[col], inplace=True)

env_cols = [c for c in df.columns if c not in ['date','Appliances','WeekStatus','Day_of_week','NSM']]
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[env_cols] = scaler.fit_transform(df[env_cols])

df_enc = pd.get_dummies(df_scaled, columns=['WeekStatus','Day_of_week'],
                        prefix=['WeekStatus','Day_of_week'], prefix_sep='')
dummy_cols = [c for c in df_enc.columns if c.startswith(('WeekStatus','Day_of_week'))]
df_enc[dummy_cols] = df_enc[dummy_cols].astype(float)
df_enc['High_Energy_Usage'] = (df_enc['Appliances'] >= 100).astype(int)

train_df, test_df = train_test_split(df_enc, test_size=0.25, random_state=1,
                                      stratify=df_enc['High_Energy_Usage'])
train_df.to_csv('training_cleaned.csv', index=False)
test_df.to_csv('testing_cleaned.csv', index=False)

print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')
print(f'High_Energy_Usage rate (train): {train_df["High_Energy_Usage"].mean()*100:.1f}%')


In [ ]:
# Correlation Heatmap
top_cols = ['Appliances','T1','RH_1','T2','RH_2','T3','RH_3',
            'T_out','RH_out','Press_mm_hg','Windspeed','Visibility','Tdewpoint','NSM','lights']
top_cols = [c for c in top_cols if c in df_enc.columns]
corr = df_enc[top_cols].corr()
plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5,
            annot_kws={'size': 8}, square=True)
plt.title('Feature Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()


---
## 📈 2. Regression Analysis — Predicting `Appliances` Energy (Wh)

In [ ]:
DROP = ['date','High_Energy_Usage','Appliances']
X_tr = train_df.drop(columns=DROP); y_tr = train_df['Appliances'].values
X_te = test_df.drop(columns=DROP);  y_te = test_df['Appliances'].values

np.random.seed(42)
ss_idx = np.random.choice(len(X_tr), size=3000, replace=False)
X_tr_ss, y_tr_ss = X_tr.iloc[ss_idx], y_tr[ss_idx]

cv3 = KFold(n_splits=3, shuffle=True, random_state=42)
print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,} samples')


In [ ]:
# 1. Simple Linear Regression
slr = LinearRegression().fit(X_tr[['T_out']], y_tr)
y_pred_slr = slr.predict(X_te[['T_out']])

# 2. Multiple Linear Regression
mlr = LinearRegression().fit(X_tr, y_tr)
y_pred_mlr = mlr.predict(X_te)

# 3. KNN Regressor - Hyperparameter Tuning
k_range = list(range(1, 21, 2))
grid_knn_r = GridSearchCV(KNeighborsRegressor(), {'n_neighbors': k_range},
                          cv=cv3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_knn_r.fit(X_tr_ss, y_tr_ss)
best_k_r = grid_knn_r.best_params_['n_neighbors']
knn_r = KNeighborsRegressor(n_neighbors=best_k_r).fit(X_tr, y_tr)
y_pred_knn_r = knn_r.predict(X_te)
print(f'Optimal KNN K (regressor) = {best_k_r}')

# KNN Tuning Curve
plt.figure(figsize=(9, 5))
plt.plot(k_range, -grid_knn_r.cv_results_['mean_test_score'],
         marker='o', color='#2b5c8f', linewidth=2)
plt.axvline(best_k_r, color='red', linestyle='--', label=f'Best K={best_k_r}')
plt.xlabel('K'); plt.ylabel('Cross-Validated RMSE')
plt.title('KNN Regressor — Hyperparameter Tuning', fontweight='bold')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

# 4. Decision Tree Regressor
grid_dt_r = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {'max_depth': [3,5,8,10,15,20], 'min_samples_split': [2,10,20]},
    cv=cv3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_dt_r.fit(X_tr_ss, y_tr_ss)
dt_r = DecisionTreeRegressor(**grid_dt_r.best_params_, random_state=42).fit(X_tr, y_tr)
y_pred_dt_r = dt_r.predict(X_te)
print(f'Optimal DT params: {grid_dt_r.best_params_}')


In [ ]:
# Regression Performance Table
reg_perf = pd.DataFrame([
    {'Model': 'Simple LM (T_out)',             'RMSE': round(np.sqrt(mean_squared_error(y_te, y_pred_slr)), 4),  'MAE': round(mean_absolute_error(y_te, y_pred_slr), 4),  'R2': round(r2_score(y_te, y_pred_slr), 4)},
    {'Model': 'Multiple LM (all features)',    'RMSE': round(np.sqrt(mean_squared_error(y_te, y_pred_mlr)), 4),  'MAE': round(mean_absolute_error(y_te, y_pred_mlr), 4),  'R2': round(r2_score(y_te, y_pred_mlr), 4)},
    {'Model': f'KNN Regressor (K={best_k_r})', 'RMSE': round(np.sqrt(mean_squared_error(y_te, y_pred_knn_r)),4), 'MAE': round(mean_absolute_error(y_te, y_pred_knn_r),4), 'R2': round(r2_score(y_te, y_pred_knn_r),4)},
    {'Model': 'Decision Tree Regressor',       'RMSE': round(np.sqrt(mean_squared_error(y_te, y_pred_dt_r)), 4), 'MAE': round(mean_absolute_error(y_te, y_pred_dt_r), 4), 'R2': round(r2_score(y_te, y_pred_dt_r), 4)},
])
print('📋 Regression Performance:')
display(reg_perf.style.highlight_min(subset=['RMSE','MAE'], color='#c6efce').highlight_max(subset=['R2'], color='#c6efce'))


In [ ]:
# Actual vs Predicted Scatter Plots
preds_r = {
    'Simple LM':           y_pred_slr,
    'Multiple LM':         y_pred_mlr,
    f'KNN (K={best_k_r})': y_pred_knn_r,
    'Decision Tree':       y_pred_dt_r,
}
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, (name, pred) in zip(axes.flatten(), preds_r.items()):
    ax.scatter(y_te, pred, alpha=0.25, color='#34495e', s=8, rasterized=True)
    ax.plot([0,800],[0,800], color='red', linestyle='--', linewidth=2, label='Perfect fit')
    ax.set_xlim(0,800); ax.set_ylim(0,800)
    ax.set_xlabel('Actual (Wh)'); ax.set_ylabel('Predicted (Wh)')
    ax.set_title(f'{name}  (R²={r2_score(y_te,pred):.3f})', fontweight='bold')
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.3)
plt.suptitle('Actual vs Predicted — Regression Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 🔵 3. Classification Analysis — Predicting `High_Energy_Usage` (≥ 100 Wh)

In [ ]:
X_tr_c = train_df.drop(columns=DROP); y_tr_c = train_df['High_Energy_Usage'].values
X_te_c = test_df.drop(columns=DROP);  y_te_c = test_df['High_Energy_Usage'].values

np.random.seed(42)
ss_idx_c  = np.random.choice(len(X_tr_c), size=1500, replace=False)
X_tr_c_ss = X_tr_c.iloc[ss_idx_c]; y_tr_c_ss = y_tr_c[ss_idx_c]

print(f'Train: {len(X_tr_c):,}  |  Test: {len(X_te_c):,}')
print(f'Class balance — 0: {(y_tr_c==0).sum():,}  1: {(y_tr_c==1).sum():,}')


In [ ]:
print('Training Logistic Regression...')
log_r = LogisticRegression(max_iter=5000, solver='lbfgs', random_state=42).fit(X_tr_c, y_tr_c)

print('Tuning KNN Classifier...')
k_range_c = list(range(1, 16, 2))
grid_knn_c = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': k_range_c},
                          cv=cv3, scoring='accuracy', n_jobs=-1)
grid_knn_c.fit(X_tr_c_ss, y_tr_c_ss)
best_k_c = grid_knn_c.best_params_['n_neighbors']
knn_c = KNeighborsClassifier(n_neighbors=best_k_c).fit(X_tr_c, y_tr_c)
print(f'  Best K = {best_k_c}')

# KNN Classifier tuning curve
plt.figure(figsize=(9,5))
plt.plot(k_range_c, grid_knn_c.cv_results_['mean_test_score'],
         marker='o', color='#2ecc71', linewidth=2)
plt.axvline(best_k_c, color='red', linestyle='--', label=f'Best K={best_k_c}')
plt.xlabel('K'); plt.ylabel('CV Accuracy')
plt.title('KNN Classifier — Hyperparameter Tuning', fontweight='bold')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

print('Tuning Decision Tree Classifier...')
grid_dt_c = GridSearchCV(DecisionTreeClassifier(random_state=42),
                         {'max_depth':[3,5,8,12,15],'min_samples_split':[2,10,20]},
                         cv=cv3, scoring='accuracy', n_jobs=-1)
grid_dt_c.fit(X_tr_c_ss, y_tr_c_ss)
dt_c = DecisionTreeClassifier(**grid_dt_c.best_params_, random_state=42).fit(X_tr_c, y_tr_c)
print(f'  Best DT params: {grid_dt_c.best_params_}')

print('Training Naive Bayes...')
nb = GaussianNB().fit(X_tr_c, y_tr_c)

print('Training SVM (RBF)...')
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_tr_c_ss, y_tr_c_ss)

print('Training Perceptron...')
perc = Perceptron(max_iter=1000, random_state=42).fit(X_tr_c, y_tr_c)

print('Training MLP Classifier...')
mlp = MLPClassifier(hidden_layer_sizes=(32,16), max_iter=200, random_state=42).fit(X_tr_c, y_tr_c)

print('\n✅ All classifiers trained.')


In [ ]:
# Evaluate all classifiers
classifiers = {
    'Logistic Regression':  (log_r.predict(X_te_c),  log_r.predict_proba(X_te_c)[:,1]),
    f'KNN (K={best_k_c})':  (knn_c.predict(X_te_c),  knn_c.predict_proba(X_te_c)[:,1]),
    'Decision Tree':        (dt_c.predict(X_te_c),   dt_c.predict_proba(X_te_c)[:,1]),
    'Naive Bayes':          (nb.predict(X_te_c),     nb.predict_proba(X_te_c)[:,1]),
    'SVM (RBF)':            (svm.predict(X_te_c),    svm.decision_function(X_te_c)),
    'Perceptron':           (perc.predict(X_te_c),   perc.decision_function(X_te_c)),
    'MLP Classifier':       (mlp.predict(X_te_c),    mlp.predict_proba(X_te_c)[:,1]),
}
clf_results = []
for name, (pred, score) in classifiers.items():
    clf_results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_te_c, pred), 4),
        'Precision': round(precision_score(y_te_c, pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_te_c, pred), 4),
        'F1-Score':  round(f1_score(y_te_c, pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_te_c, score), 4),
    })
clf_perf = pd.DataFrame(clf_results)
print('📋 Classification Performance:')
display(clf_perf.style.highlight_max(subset=['Accuracy','F1-Score','ROC-AUC'], color='#c6efce'))


In [ ]:
# ROC Curves
colors_roc = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf']
fig, ax = plt.subplots(figsize=(11, 8))
for (name,(pred,score)), color in zip(classifiers.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_te_c, score)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_te_c,score):.3f})', linewidth=2, color=color)
ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate',fontsize=12); ax.set_ylabel('True Positive Rate',fontsize=12)
ax.set_title('ROC Curves — All Classifiers', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=9); ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Confusion Matrices Grid
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes_flat = axes.flatten()
for i, (name,(pred,_)) in enumerate(classifiers.items()):
    cm = confusion_matrix(y_te_c, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_flat[i], cbar=False, linewidths=0.5)
    axes_flat[i].set_title(name, fontweight='bold', fontsize=11)
    axes_flat[i].set_xlabel('Predicted'); axes_flat[i].set_ylabel('Actual')
    axes_flat[i].set_xticklabels(['Normal','High'], fontsize=9)
    axes_flat[i].set_yticklabels(['Normal','High'], fontsize=9, rotation=0)
for j in range(len(classifiers), len(axes_flat)): fig.delaxes(axes_flat[j])
plt.suptitle('Confusion Matrices — All Classifiers', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# SVM Decision Boundary in 2D PCA Space
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_tr_c_ss)
svm_2d = SVC(kernel='rbf', C=1.0, random_state=42).fit(X_pca, y_tr_c_ss)

h = 0.15
x_min,x_max = X_pca[:,0].min()-0.5, X_pca[:,0].max()+0.5
y_min,y_max = X_pca[:,1].min()-0.5, X_pca[:,1].max()+0.5
xx,yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
Z = svm_2d.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(11,8))
ax.contourf(xx,yy,Z,cmap=plt.cm.coolwarm,alpha=0.3)
sc = ax.scatter(X_pca[:,0],X_pca[:,1],c=y_tr_c_ss,cmap=plt.cm.coolwarm,edgecolors='k',s=25,alpha=0.7)
svs = svm_2d.support_vectors_
ax.scatter(svs[:,0],svs[:,1],s=100,facecolors='none',edgecolors='black',linewidths=1.8,label='Support Vectors')
ax.set_xlabel('PC1',fontsize=12); ax.set_ylabel('PC2',fontsize=12)
ax.set_title('SVM Decision Boundary & Support Vectors (2D PCA)', fontweight='bold', fontsize=13)
ax.legend(); ax.grid(True,alpha=0.2)
plt.colorbar(sc, ax=ax, label='Class (0=Normal, 1=High)')
plt.tight_layout(); plt.show()


---
## 🔶 4. Clustering Analysis — Indoor Environmental Segmentation (T1–T9, RH_1–RH_9)

In [ ]:
CLUST_FEATURES = [c for c in ['T1','RH_1','T2','RH_2','T3','RH_3','T4','RH_4',
                               'T5','RH_5','T6','RH_6','T7','RH_7','T8','RH_8','T9','RH_9']
                  if c in train_df.columns]
X_clust = train_df[CLUST_FEATURES]

np.random.seed(42)
ss_idx_cl = np.random.choice(len(X_clust), size=3000, replace=False)
X_cl_ss   = X_clust.iloc[ss_idx_cl]

print(f'Clustering dataset: {X_clust.shape}  |  Subsample: {X_cl_ss.shape}')


In [ ]:
# K-Means Tuning — Elbow & Silhouette
cluster_range = list(range(2, 9))
inertias, sil_scores, db_scores = [], [], []

for k in cluster_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cl_ss)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cl_ss, km.labels_))
    db_scores.append(davies_bouldin_score(X_cl_ss, km.labels_))
    print(f'K={k}  Silhouette={sil_scores[-1]:.3f}  Davies-Bouldin={db_scores[-1]:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(cluster_range, inertias, marker='o', color='#34495e', linewidth=2)
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method', fontweight='bold'); ax1.grid(True, linestyle='--', alpha=0.5)
ax2.plot(cluster_range, sil_scores, marker='o', color='#9b59b6', linewidth=2)
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score Profile', fontweight='bold'); ax2.grid(True, linestyle='--', alpha=0.5)
plt.suptitle('K-Means Cluster Tuning', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Fit final K-Means (K=3)
OPTIMAL_K = 3
km_final  = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10).fit(X_clust)
km_labels = km_final.labels_
print(f'K-Means (K={OPTIMAL_K}) fitted on {len(X_clust):,} samples.')


In [ ]:
# Hierarchical Clustering — Dendrogram
X_dendro = X_cl_ss.iloc[:150]
linkage_matrix = linkage(X_dendro, method='ward')

plt.figure(figsize=(14, 7))
dendrogram(linkage_matrix, labels=X_dendro.index.astype(str), leaf_rotation=90, leaf_font_size=7)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage, 150 samples)', fontsize=13, fontweight='bold')
plt.xlabel('Sample Index'); plt.ylabel('Ward Distance')
plt.tight_layout(); plt.show()

agg = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward')
agg_labels_ss = agg.fit_predict(X_cl_ss)
print(f'Agglomerative clustering complete on subsample.')


In [ ]:
# GMM / EM — BIC & AIC Tuning
bic_scores, aic_scores = [], []
comp_range = list(range(2, 9))
for n in comp_range:
    gmm = GaussianMixture(n_components=n, random_state=42).fit(X_cl_ss)
    bic_scores.append(gmm.bic(X_cl_ss))
    aic_scores.append(gmm.aic(X_cl_ss))

plt.figure(figsize=(9, 5))
plt.plot(comp_range, bic_scores, marker='o', color='#e74c3c', linewidth=2, label='BIC')
plt.plot(comp_range, aic_scores, marker='o', color='#3498db', linewidth=2, label='AIC')
plt.xlabel('Number of GMM Components', fontsize=12)
plt.ylabel('Information Criterion Score', fontsize=12)
plt.title('GMM Components Tuning (AIC / BIC)', fontweight='bold', fontsize=13)
plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

gmm_final  = GaussianMixture(n_components=OPTIMAL_K, random_state=42).fit(X_clust)
gmm_labels = gmm_final.predict(X_clust)
print(f'GMM EM (components={OPTIMAL_K}) fitted.')


In [ ]:
# Clustering Performance Table
km_labels_ss  = km_labels[ss_idx_cl]
gmm_labels_ss = gmm_labels[ss_idx_cl]

clust_perf = pd.DataFrame({
    'Metric':              ['Silhouette Score (↑ better)', 'Davies-Bouldin Index (↓ better)'],
    'K-Means':             [round(silhouette_score(X_cl_ss, km_labels_ss),4),  round(davies_bouldin_score(X_cl_ss, km_labels_ss),4)],
    'Hierarchical (Ward)': [round(silhouette_score(X_cl_ss, agg_labels_ss),4), round(davies_bouldin_score(X_cl_ss, agg_labels_ss),4)],
    'GMM (EM)':            [round(silhouette_score(X_cl_ss, gmm_labels_ss),4),  round(davies_bouldin_score(X_cl_ss, gmm_labels_ss),4)],
})
print('📋 Clustering Performance Comparison:')
display(clust_perf)


In [ ]:
# 3-Panel PCA Cluster Projection
pca_cl = PCA(n_components=2)
X_clust_pca = pca_cl.fit_transform(X_clust)
X_cl_ss_pca = pca_cl.transform(X_cl_ss)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 6))

for ax, labels, title in [
    (ax1, km_labels,  f'K-Means (K={OPTIMAL_K})'),
    (ax2, gmm_labels, f'GMM EM (Components={OPTIMAL_K})'),
]:
    sns.scatterplot(x=X_clust_pca[:,0], y=X_clust_pca[:,1],
                    hue=labels, palette='viridis', ax=ax, s=12, alpha=0.5, legend='full')
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

sns.scatterplot(x=X_cl_ss_pca[:,0], y=X_cl_ss_pca[:,1],
                hue=agg_labels_ss, palette='viridis', ax=ax3, s=20, alpha=0.7, legend='full')
ax3.set_title(f'Hierarchical Ward (K={OPTIMAL_K}) — subsample', fontweight='bold', fontsize=13)
ax3.set_xlabel('PC1'); ax3.set_ylabel('PC2')

plt.suptitle('2D PCA Projection — Cluster Assignments', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## ✅ 5. Final Summary — All Model Performances

In [ ]:
print('=' * 70)
print('       APPLIANCES ENERGY PREDICTION — COMPLETE RESULTS SUMMARY')
print('=' * 70)
print('\n📈 REGRESSION  (Target: Appliances Wh)')
display(reg_perf)
print('\n🔵 CLASSIFICATION  (Target: High_Energy_Usage ≥ 100 Wh)')
display(clf_perf)
print('\n🔶 CLUSTERING  (Features: Indoor Temperature & Humidity T1-T9, RH_1-RH_9)')
display(clust_perf)
print('\n' + '=' * 70)
